# GEPA

Use textual failure feedback to evolve instructions, preserving reflection logs and best outputs along the way.

**Use it when:** Your metric can explain errors, not merely score them, and you want a detailed instruction that encodes those lessons.

**What compilation changes:** Instruction text and, depending on the search path, demonstrations; inspect the frozen reference artifact.

Important configuration in this benchmark:

- six full evaluations in the full profile
- reflection minibatches of three
- feedback metric returns both score and diagnosis
- seed 42; merge disabled for a bounded run

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'gepa'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='gepa'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.GEPA(
    metric=feedback_metric, max_full_evals=profile.gepa_max_full_evals,
    reflection_minibatch_size=3, reflection_lm=reflection_lm,
    use_merge=False, track_best_outputs=True, seed=42,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: gepa
task model: openai/gpt-5.6-luna
final test accuracy: 76.2% (61/80)
optimized validation accuracy: 88.3%
same-model baseline: 50.0%
uplift: +26.2 points
optimization cost: $10.8412
optimization time: 2023.1s
mean inference latency: 2.060s
p95 inference latency: 3.125s

Published artifacts:
- program snapshot: chapter06/results/gepa_expanded/final/optimized_program.json
- prompt snapshot: chapter06/results/gepa_expanded/final/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The preview is deliberately truncated; follow the prompt path for the complete learned rule set and `optimizer_logs/` for the evolutionary trace.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (13438 characters):
Determine whether a supplied passage was more likely generated by AI or substantially rewritten by AI.

Input format
- `text`: one passage to evaluate.

Output format
Return exactly these two fields and no others:
- `reasoning`: A concise, evidence-based comparative provenance judgment.
- `is_ai`: Boolean `True` if AI generation or substantial AI rewriting is more likely; otherwise `False`.

Do not claim certainty. Style-based provenance detection is probabilistic, and no single feature proves authorship. Evaluate the interacting pattern across the whole passage.

Core comparison

Compare:

1. Local source texture
Evidence that the passage was composed sentence by sentence for an immediate documentation, tutorial, academic, historical, or practical purpose:

- Exact APIs, identifiers, variables, versioned names, encodings, citations, historical references, workflow steps, and canonical terminology
- Direct commands, abru

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.